# 01 · Exploratory data analysis — PlantVillage + PlantDoc

What this notebook answers, and the numbers that go into the report:
1. How imbalanced are the 38 classes?
2. What are the image sizes and are they consistent?
3. Are there exact duplicate images between train and valid (leakage)?
4. How different does PlantDoc look from PlantVillage? (this is the honest-framing slide)


In [ ]:
# Kaggle setup — run this first in every notebook.
# Settings -> Accelerator: GPU T4 x2 | Internet: ON
import os, subprocess, sys
REPO = "/kaggle/working/SKIT-DS-2023-2027-015"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/shankerjoshi/SKIT-DS-2023-2027-015.git", REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime", "onnxconverter-common", "shap"], check=True)
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


In [ ]:
from data.download import kaggle_download
from pathlib import Path
kaggle_download("vipoooool/new-plant-diseases-dataset", Path("data/raw/plantvillage"))
kaggle_download("nirmalsankalana/plantdoc-dataset", Path("data/raw/plantdoc"))
kaggle_download("atharvaingle/crop-recommendation-dataset", Path("data/raw/crop"))


In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
from PIL import Image
from collections import Counter

PV = Path("data/raw/plantvillage/New Plant Diseases Dataset(Augmented)")
train_dir, valid_dir = PV / "train", PV / "valid"
rows = []
for split, root in (("train", train_dir), ("valid", valid_dir)):
    for p in root.rglob("*"):
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
            rows.append({"split": split, "class": p.parent.name, "path": str(p), "size_bytes": p.stat().st_size})
df = pd.DataFrame(rows)
print(df.split.value_counts())
print("classes:", df["class"].nunique(), "| total images:", len(df))


In [ ]:
import matplotlib.pyplot as plt
counts = df[df.split == "train"]["class"].value_counts()
print("most common:", counts.head(3).to_dict())
print("rarest:", counts.tail(3).to_dict())
print("imbalance ratio:", round(counts.max() / counts.min(), 2))

fig, ax = plt.subplots(figsize=(10, 10))
counts.sort_values().plot.barh(ax=ax)
ax.set_xlabel("training images"); ax.set_title("PlantVillage class distribution")
plt.tight_layout(); plt.savefig("results_eda_class_balance.png", dpi=110); plt.show()


In [ ]:
# image size + duplicate check (md5 catches the augmented copies Kaggle ships)
import hashlib
sizes = Counter()
hashes = Counter()
for p in df[df.split == "train"]["path"].sample(2000, random_state=0):
    with Image.open(p) as im:
        sizes[im.size] += 1
    hashes[hashlib.md5(open(p, "rb").read()).hexdigest()] += 1
print("distinct sizes:", sizes.most_common(5))
print("duplicate rate in the 2000 sampled:", round(1 - len(hashes) / 2000, 4))


In [ ]:
# side-by-side: lab image vs field image. Put this figure in the report.
import random
pv_files = df[df.split == "train"].sample(6, random_state=1)["path"].tolist()
pd_files = [str(p) for p in Path("data/raw/plantdoc").rglob("*") if p.suffix.lower() in {".jpg", ".png"}][:6]

fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for ax, f in zip(axes[0], pv_files):
    ax.imshow(Image.open(f)); ax.set_title("lab", fontsize=8); ax.axis("off")
for ax, f in zip(axes[1], pd_files):
    ax.imshow(Image.open(f)); ax.set_title("field", fontsize=8); ax.axis("off")
plt.suptitle("PlantVillage (top) vs PlantDoc (bottom) — why two accuracy numbers are reported")
plt.tight_layout(); plt.savefig("results_eda_lab_vs_field.png", dpi=110); plt.show()


In [ ]:
# crop recommendation table
crop_csv = next(Path("data/raw/crop").rglob("*.csv"))
c = pd.read_csv(crop_csv)
print(c.shape); print(c["label"].nunique(), "crops"); print(c.describe().round(2))
print(c["label"].value_counts().describe())


### Findings to carry into the report
- Class imbalance ratio and the rarest classes (these drive the class-weight choice in training).
- Whether duplicate images exist between train and valid — if yes, say so and use the shipped split anyway, noting the caveat.
- The lab-vs-field figure justifies reporting two accuracy numbers.
